# LiteRT-LM Unified NPU Export & Quantization Guide

This Colab demonstrates the unified, end-to-end Python workflow for exporting, calibrating, statically range quantizing (`A16W8`), and compiling conversational LLMs into `.litertlm` bundles **targeting LiteRT NPU backends**.

## Prerequisites: Open-Source Installation
Install open-source LiteRT, LiteRT-Torch, and HuggingFace PyTorch dependencies using one of the options below:

### Option A: Install Nightly Release (Recommended)
```bash
!pip install -q --pre ai-edge-litert litert-torch-nightly transformers
```

### Option B: Install Directly from GitHub Main Branch
```bash
!pip install -q git+https://github.com/google-ai-edge/litert-torch.git transformers
```

### NPU Compiler SDK Setup (Required for Stage 4 Compilation)
To compile models into `.litertlm` bundles targeting specific NPU SoCs (`sm8850`, `mt6993`), install the optional vendor SDK for your target hardware:
```bash
# For Qualcomm Snapdragon SoCs (QNN):
# !pip install ai-edge-litert-sdk-qualcomm

# For MediaTek Dimensity SoCs (Neuron):
# !pip install ai-edge-litert-sdk-mediatek
```


## 1. Hierarchical Pipeline Configuration
Enter the target **Model ID** (`google/gemma-3-270m-it`, `google/gemma-3-1b-it`, `Qwen/Qwen3-0.6B`), select the **Weight Quantization Recipe** (`dynamic_wi8_afp32`), **Target Vendor** (`qualcomm` or `mediatek`), and type the **Target SoC** chip model in the text box (`sm8850`, `sw6100`, `mt6993`, etc.). The configuration manager dynamically validates the SoC against the authoritative `supported_soc.csv` for that vendor and merges hierarchical defaults (`Vendor -> SoC -> User Overrides`).

In [ ]:
# @title Hierarchical Pipeline Configuration
import os
from litert_torch.generative.export_hf.experimental.npu_export.config_manager import build_pipeline_config

# Recommended options: "google/gemma-3-270m-it", "google/gemma-3-1b-it", "Qwen/Qwen3-0.6B"
SELECT_MODEL = "meta-llama/Llama-3.2-1B"  # @param {type:"string"}
WEIGHT_QUANTIZATION_RECIPE = "dynamic_wi8_afp32"  # @param ["dynamic_wi8_afp32"]
TARGET_VENDOR = "qualcomm"  # @param ["qualcomm", "mediatek"]
TARGET_SOC = "sm8850"  # @param {type:"string"}

# Prefill bucket length (tokens). Note: Restricted to 128 due to current NPU runtime limitations.
PREFILL_LENGTH = 128  # @param [128]
CACHE_LENGTH = 1024  # @param {type:"integer"}

# Number of sample prompts (from the 50 randomly generated experimental prompts) to use for calibration.
# Note: For production quality, bring your own domain-representative quality-proofed calibration dataset.
CALIB_PROMPTS_COUNT = 50  # @param {type:"integer"}

# Token generation steps per prompt during calibration.
# Longer steps give better activation bound accuracy but increase calibration time.
# Recommended: 32 for fast experimental iteration; 128+ for production deployments.
CALIB_MAX_DECODE_STEPS = 32  # @param {type:"integer"}

cfg = build_pipeline_config(
    model_id=SELECT_MODEL,
    target_vendor=TARGET_VENDOR,
    target_soc=TARGET_SOC,
    weight_quantization_recipe=WEIGHT_QUANTIZATION_RECIPE,
    prefill_lengths=[PREFILL_LENGTH],
    cache_length=CACHE_LENGTH,
    max_decode_steps=CALIB_MAX_DECODE_STEPS,
)

# -----------------------------------------------------------------------------
# Centralized Pipeline Workspace Paths & Directory Setup
# -----------------------------------------------------------------------------
clean_model_name = (
    cfg.model_id.split("/")[-1].replace("-", "_").replace(".", "_").lower()
)
WORK_DIR = (
    f"/tmp/litert_npu_pipeline_{clean_model_name}_{cfg.backend}_{cfg.soc_model}"
)
EXPORT_DIR = os.path.join(WORK_DIR, "exported")
CALIB_DATA_DIR = os.path.join(WORK_DIR, "calib_data")
CALIB_OUT_DIR = os.path.join(WORK_DIR, "calib_bounds")
SRQ_DIR = os.path.join(WORK_DIR, "srq_quantized")
NPU_DIR = os.path.join(WORK_DIR, "npu_compiled")

for d in [
    WORK_DIR,
    EXPORT_DIR,
    CALIB_DATA_DIR,
    CALIB_OUT_DIR,
    SRQ_DIR,
    NPU_DIR,
]:
  os.makedirs(d, exist_ok=True)

# Pre-define key artifact paths across the pipeline stages:
EXPORTED_LITERTLM = os.path.join(EXPORT_DIR, "model.litertlm")
SRQ_LITERTLM = os.path.join(SRQ_DIR, "model_srq.litertlm")
NPU_LITERTLM = os.path.join(
    NPU_DIR, f"model_{cfg.backend}_{cfg.soc_model}.litertlm"
)

print("=== Pipeline Workspace & Directories Initialized ===")
print(f"  Root Working Directory: {WORK_DIR}")
print(f"  Stage 1 Exported      : {EXPORTED_LITERTLM}")
print(f"  Stage 2 Calib Bounds  : {CALIB_OUT_DIR}")
print(f"  Stage 3 SRQ Bundle    : {SRQ_LITERTLM}")
print(f"  Stage 4 NPU Bundle    : {NPU_LITERTLM}\n")

cfg.print_summary()

### [Optional] Advanced Custom Pipeline Overrides & JSON Serialization
Uncomment any lines below in Code Cell 4 to manually customize individual parameters (`model_id`, `weight_quantization_recipe`, `use_16bits_activations`, `allow_float_operations`, etc.) or to **save/load your configuration to/from a JSON file** (`save_json` / `load_json`) after the hierarchical defaults load:

In [ ]:
# @title [Optional] Advanced Custom Pipeline Overrides & JSON Serialization
# --- Option A: Manual Property Overrides ---
# Uncomment any lines below to manually override specific pipeline properties:
# cfg.model_id = "google/gemma-3-1b-it"
# cfg.weight_quantization_recipe = "dynamic_wi4_afp32"
# cfg.use_16bits_activations = True
# cfg.allow_float_operations = False

# --- Option B: Save or Load Configuration from JSON ---
# Save your resolved configuration snapshot to JSON to carry across jobs/scripts:
# cfg.save_json(os.path.join(WORK_DIR, "pipeline_config.json"))

# Or load a pre-saved configuration directly from JSON (overriding the form defaults above):
# from litert_torch.generative.export_hf.experimental.npu_export.config_manager import NpuPipelineConfig
# cfg = NpuPipelineConfig.load_json("/path/to/your/saved_pipeline_config.json")

# If you loaded from JSON or modified model_id/backend/soc_model above, re-sync paths:
# clean_model_name = cfg.model_id.split("/")[-1].replace("-", "_").replace(".", "_").lower()
# WORK_DIR = f"/tmp/litert_npu_pipeline_{clean_model_name}_{cfg.backend}_{cfg.soc_model}"
# EXPORT_DIR, CALIB_DATA_DIR, CALIB_OUT_DIR, SRQ_DIR, NPU_DIR = [os.path.join(WORK_DIR, sub) for sub in ["exported", "calib_data", "calib_bounds", "srq_quantized", "npu_compiled"]]
# for d in [WORK_DIR, EXPORT_DIR, CALIB_DATA_DIR, CALIB_OUT_DIR, SRQ_DIR, NPU_DIR]: os.makedirs(d, exist_ok=True)
# EXPORTED_LITERTLM = os.path.join(EXPORT_DIR, "model.litertlm")
# SRQ_LITERTLM = os.path.join(SRQ_DIR, "model_srq.litertlm")
# NPU_LITERTLM = os.path.join(NPU_DIR, f"model_{cfg.backend}_{cfg.soc_model}.litertlm")

# print("=== Updated Configuration Summary ===")
# cfg.print_summary()

## 2. Stage 1: Export (`export`)
Export the selected HuggingFace PyTorch model (`cfg.model_id`) using the weight quantization recipe (`cfg.weight_quantization_recipe`) and vendor-specific attention time-step dimensions (`cfg.k_ts_idx`, `cfg.v_ts_idx`).

In [ ]:
from litert_torch.generative.export_hf.experimental.npu_export import stages

print(
    f"=== Stage 1: Exporting {cfg.model_id}"
    f" ({cfg.weight_quantization_recipe}) ==="
)
stages.npu_export(cfg=cfg, output_dir=EXPORT_DIR)
print(f"Exported bundle ready at: {EXPORTED_LITERTLM}")

## 3. Stage 2: Calibration (`calibrate`)
Load calibration prompts from the built-in 50-prompt experimental dataset (`sample_calibration_prompts.json`).

> **Production Guidance & Mandatory Prompt Length Requirements:**
> When preparing your own domain-representative calibration dataset (`in production or custom workflows`), **you must include longer prompts (`e.g., prompts whose token count strictly exceeds your configured `prefill_lengths` bucket, such as > 200–300+ tokens for a 128-token prefill bucket`)**.
> 
> **Why is this required?**
> Prompts longer than the prefill bucket length (`128 tokens`) force the `prefill_128` signature to execute **multiple turns across the prompt**:
> 1. **Turn 0 (First 128 tokens):** `prefill_128` runs with zeroed initial `kv_cache_k` / `kv_cache_v` inputs, meaning historical attention branches (`BATCH_MATMUL` scores and residual `ADD` operations) evaluate to `0.0`.
> 2. **Turn 1 (Subsequent overflow tokens):** `prefill_128` runs again with `prefill_128_kv_cache_v_{layer}` populated with historical keys/values from Turn 0. This populates real, non-zero dynamic range statistics (`stats.min`, `stats.max`) across all historical attention nodes during profiler calibration, preventing zero-collapse (`[0.0, 0.0]`) and multiplier shift overflow errors (`shift >= 8`) right during execution.

In [ ]:
import json
from litert_torch.generative.export_hf.experimental.npu_export import stages

sample_json_path = "third_party/py/litert_torch/generative/export_hf/experimental/npu_export/sample_calibration_prompts.json"
if os.path.exists(sample_json_path):
  with open(sample_json_path, "r") as f:
    all_prompts = json.load(f)
else:
  # Fallback prompts (ensure they exceed 128 tokens so multi-turn prefill_128 populates historical KV-cache statistics)
  all_prompts = [
      (
          "Critically evaluate the historical and astronomical accuracy of the"
          " widespread claim that the Great Wall of China is clearly visible to"
          " the naked human eye from the surface of the Moon or low Earth"
          " orbit. In your detailed assessment, provide specific optical"
          " calculations regarding the angular resolution of the human eye, the"
          " typical width of the wall in meters, the atmospheric contrast"
          " conditions, and the orbital distance involved. Furthermore, trace"
          " the origin of this misconception back to early 20th-century"
          " writings that predated human spaceflight, and contrast these"
          " historical claims with documented statements from astronauts and"
          " cosmonauts who have actually observed the Earth from both space"
          " shuttles and the International Space Station across varying weather"
          " conditions and atmospheric visibility profiles."
      ),
      (
          "A high-speed passenger train departs from Terminal Station Alpha"
          " traveling eastbound at a constant velocity of 65 miles per hour"
          " toward Terminal Station Beta, which is located exactly 315 miles"
          " away along a direct, uninterrupted parallel rail line. Exactly"
          " forty-five minutes later, an express freight train departs from"
          " Terminal Station Beta heading westbound on the adjacent track"
          " toward Terminal Station Alpha at a constant velocity of 85 miles"
          " per hour. Calculate precisely how many hours and minutes will"
          " elapse from the moment the passenger train departs until the front"
          " engines of the two trains pass each other, and determine the exact"
          " geographical distance in miles from Terminal Station Alpha to that"
          " meeting point across uniform motion formulas."
      ),
      (
          "Write a robust, production-grade Python function using the"
          " two-pointer algorithmic technique to determine whether a given"
          " UTF-8 string is a valid palindrome while strictly ignoring all"
          " punctuation marks, whitespace, diacritics, and case distinctions"
          " across international character sets. Your implementation must"
          " include comprehensive type hints from the typing module, explicit"
          " edge-case handling for empty strings or strings containing solely"
          " symbols, and inline comments detailing every step. In addition,"
          " provide a comprehensive time and space complexity analysis"
          " explaining why this two-pointer approach operates in guaranteed"
          " O(N) linear time and O(1) auxiliary memory compared to naïve string"
          " reversal strategies across high-performance applications."
      ),
  ]

selected_prompts = all_prompts[:CALIB_PROMPTS_COUNT]
print(
    f"Loading {len(selected_prompts)} sample prompts for experimental"
    " calibration..."
)
print(
    "Note: For production quality, provide your own quality-proofed calibration"
    " dataset."
)
print(
    "IMPORTANT: Ensure your dataset includes prompts longer than the prefill"
    " bucket length (128 tokens)"
)
print(
    "           to force multi-turn prefill execution and accurately calibrate"
    " historical KV-cache attention ranges."
)

with open(os.path.join(CALIB_DATA_DIR, "ALL.jsonl"), "w") as f:
  for p in selected_prompts:
    f.write(json.dumps({"text": p}) + "\n")

print("\n--- Stage 2: Running Profiler-Based Calibration ---")
stages.npu_calibrate(
    cfg=cfg,
    input_litertlm=EXPORTED_LITERTLM,
    dataset_dir=CALIB_DATA_DIR,
    calibration_result_save_dir=CALIB_OUT_DIR,
)
print(f"Calibration completed. Profiles stored at: {CALIB_OUT_DIR}")

## 4. Stage 3: Static Range Quantization (`quantize`)
Statically range quantize the intermediate bundle (`cfg.use_16bits_activations`, `cfg.allow_float_operations`). Passing `calibration_dir=CALIB_OUT_DIR` automatically resolves both main (`prefill_decode`) and auxiliary (`aux`) profiles.

In [ ]:
from litert_torch.generative.export_hf.experimental.npu_export import stages

print(
    "--- Stage 3: Static Range Quantizing"
    f" (use_16bits_activations={cfg.use_16bits_activations},"
    f" allow_float_ops={cfg.allow_float_operations}) ---"
)
# cfg.allow_float_operations = True
stages.npu_quantize(
    cfg=cfg,
    input_litertlm=EXPORTED_LITERTLM,
    calibration_dir=CALIB_OUT_DIR,  # Automatically resolves prefill_decode and aux profile JSONs
    output_litertlm=SRQ_LITERTLM,
)
print(f"5-Section SRQ Bundle created at: {SRQ_LITERTLM}")

## 5. Stage 4: NPU Target Compilation (`compile_litertlm`)
Compile the statically quantized model into native NPU hardware binaries for the automatically derived vendor backend (`cfg.backend`) and SoC (`cfg.soc_model`).

In [ ]:
from litert_torch.generative.export_hf.experimental.npu_export import stages

print(f"=== Stage 4: Compiling for {cfg.backend.upper()} ({cfg.soc_model}) ===")
stages.npu_compile(
    cfg=cfg,
    input_litertlm=SRQ_LITERTLM,
    output_litertlm=NPU_LITERTLM,
)

print(f"NPU Bundle ready for on-device deployment at: {NPU_LITERTLM}")

## 6. Bundle Summary & On-Device Deployment
Inspect the 5 sections of the compiled NPU `.litertlm` bundle and note deployment instructions for LiteRT-LM runtime (`--backend=npu`).

In [ ]:
from litert_torch.generative.export_hf.experimental.litertlm_bundle import litertlm_bundle as litertlm_utils

print("=== NPU Bundle Sections ===")
print(litertlm_utils.peek_litertlm(NPU_LITERTLM))

print("\n=== On-Device Deployment Guidance ===")
print(
    f"Bundle compiled for {cfg.backend.upper()} ({cfg.soc_model}) at:"
    f" {NPU_LITERTLM}"
)
print(
    "Push and execute this model on your target device using LiteRT-LM runtime"
    " with `--backend=npu`."
)